# Recommender comparison over a neuroscience / cognitive-science library
**Honest, reproducible pipeline** for the project *"Porownanie skutecznosci modeli rekomendacyjnych..."*.

This notebook runs end to end and generates every number itself, which is the piece the original project was missing. It compares two content-based recommenders, TF-IDF (lexical) and Sentence-Transformers `all-MiniLM-L6-v2` (semantic), against two baselines, on a library of 36,314 arXiv papers.

**The key design point (why this version is fair).** Each synthetic reader's ground truth (their relevant set) is defined by arXiv **category co-membership** (q-bio.NC AND a second field). Categories are author-assigned metadata; neither model reads them. Both models only see title + abstract text. So the answer key was not written with either model's method, and the comparison is not circular.

Inputs (in `data/`): `corpus.jsonl.gz` (id, title, abstract, categories) and `users.json` (62 readers with seed history + held-out ground truth). Run the cells top to bottom. The MiniLM encoding of 36k abstracts is the one slow step (a few minutes on CPU, seconds on GPU).

In [ ]:
# 1. Dependencies (safe to re-run)
%pip install -q "numpy<2" scikit-learn scipy pandas sentence-transformers

In [ ]:
# 2. Imports and config
import os, json, gzip, time, math, random
import numpy as np, pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED)
KS = [5, 10]          # cutoffs for Precision/Recall/NDCG@K
DATA = "data"
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

In [ ]:
# 3. Load the corpus and the synthetic readers
def load_corpus(path):
    ids, texts, cats = [], [], []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            ids.append(r["id"])
            texts.append(((r.get("title","") or "") + ". " + (r.get("abstract","") or "")).strip())
            cats.append(r.get("categories",""))
    return ids, texts, cats

ids, texts, cats = load_corpus(f"{DATA}/corpus.jsonl.gz")
pos = {p:i for i,p in enumerate(ids)}
N = len(ids)
users = json.load(open(f"{DATA}/users.json"))["users"]
print(f"{N} papers, {len(users)} readers")
print("ground-truth signal:", json.load(open(f"{DATA}/users.json"))["ground_truth_signal"])

## Engine 1 — TF-IDF (lexical baseline)
A reader is represented by the mean TF-IDF vector of their 8 seed papers. Candidates are ranked by cosine similarity. Their own seed papers are excluded from the ranking.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
t = time.time()
vec = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5,
                      ngram_range=(1,2), max_features=50000, sublinear_tf=True)
X = vec.fit_transform(texts)          # N x V, L2-normalised rows (norm='l2' default)
build_tfidf = time.time() - t
print(f"TF-IDF index: {X.shape[1]} features, built in {build_tfidf:.1f}s")

def recommend_tfidf(profile_ids, k, exclude):
    idx = [pos[p] for p in profile_ids if p in pos]
    prof = np.asarray(X[idx].mean(axis=0)).ravel()          # mean of seed vectors
    sims = X.dot(prof)                                       # cosine up to a per-query constant
    sims[list(exclude)] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    return top[np.argsort(-sims[top])]

## Engine 2 — Sentence-Transformers `all-MiniLM-L6-v2` (semantic)
Same protocol, but papers are represented by 384-dim sentence embeddings. This cell is the slow one.

In [ ]:
# ---- Engine 2: MiniLM (GPU/MPS if available; embeddings cached for instant re-runs) ----
EMB_CACHE = "data/emb_minilm.npy"
emb = None; build_minilm = 0.0
if os.path.exists(EMB_CACHE):
    try:
        if np.load(EMB_CACHE, mmap_mode="r").shape[0] == N:
            emb = np.load(EMB_CACHE)
            print(f"loaded cached MiniLM embeddings {emb.shape} (delete {EMB_CACHE} to re-encode)")
    except Exception:
        emb = None
if emb is None:
    import torch
    from sentence_transformers import SentenceTransformer
    dev = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"encoding with device = {dev}")
    model = SentenceTransformer("all-MiniLM-L6-v2", device=dev)
    t = time.time()
    emb = model.encode(texts, batch_size=64, show_progress_bar=True,
                       normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    build_minilm = time.time() - t
    np.save(EMB_CACHE, emb)
    print(f"MiniLM embeddings {emb.shape} encoded + cached in {build_minilm:.1f}s")

def recommend_minilm(profile_ids, k, exclude):
    idx = [pos[p] for p in profile_ids if p in pos]
    prof = emb[idx].mean(axis=0)
    sims = emb.dot(prof)
    sims[list(exclude)] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    return top[np.argsort(-sims[top])]

## Baselines — a floor to measure against
**Random**: a random ranking. **Popularity**: the same non-personalised list for everyone, ranking papers by how prevalent their categories are in the corpus. If a personalised model cannot beat "recommend the broadly-central papers to everyone", it is not adding much.

In [ ]:
from collections import Counter
catf = Counter()
for c in cats:
    for x in set(c.split()): catf[x] += 1
popscore = np.array([sum(catf[x] for x in set(cats[i].split())) for i in range(N)])
pop_order = list(np.argsort(-popscore))
_rng = random.Random(SEED)

def recommend_popularity(k, exclude):
    return [d for d in pop_order if d not in exclude][:k]

def recommend_random(k, exclude):
    return _rng.sample([d for d in range(N) if d not in exclude], k)

## Evaluation — Precision@K, Recall@K, NDCG@K, per reader
Metrics are computed per reader and then averaged, so we keep the per-reader distribution for the significance test.

In [ ]:
def ndcg(hits, k, n_gt):
    dcg  = sum(1/math.log2(i+2) for i,h in enumerate(hits[:k]) if h)
    idcg = sum(1/math.log2(i+2) for i in range(min(n_gt, k))) or 1.0
    return dcg/idcg

def evaluate(name, recommender, uses_profile=True):
    rows = []
    t = time.time()
    for u in users:
        gt   = set(pos[p] for p in u["ground_truth_ids"] if p in pos)
        excl = set(pos[p] for p in u["profile_ids"]      if p in pos)
        rec  = recommender(u["profile_ids"], max(KS), excl) if uses_profile else recommender(max(KS), excl)
        rec  = list(rec)
        hits = [1 if d in gt else 0 for d in rec]
        row  = {"reader": u["user_id"]}
        for k in KS:
            row[f"P@{k}"]    = sum(hits[:k])/k
            row[f"R@{k}"]    = sum(hits[:k])/max(len(gt),1)
            row[f"NDCG@{k}"] = ndcg(hits, k, len(gt))
        rows.append(row)
    df = pd.DataFrame(rows); df["engine"] = name
    return df, time.time()-t

detail = {}
detail["TF-IDF"],     qt_tfidf  = evaluate("TF-IDF",     recommend_tfidf)
detail["MiniLM"],     qt_minilm = evaluate("MiniLM",     recommend_minilm)
detail["Popularity"], _         = evaluate("Popularity", recommend_popularity, uses_profile=False)
detail["Random"],     _         = evaluate("Random",     recommend_random,     uses_profile=False)

summary = (pd.concat(detail.values())
             .groupby("engine")[[f"{m}@{k}" for m in ("P","R","NDCG") for k in KS]]
             .mean()
             .loc[["Random","Popularity","TF-IDF","MiniLM"]])
print("Mean metrics over", len(users), "readers (honest category ground truth):")
summary

## Significance and timing (hypotheses H1 and H2)
**H1** (semantic beats lexical) is a claim about variance across readers, so it needs a paired test, not just a gap in means. **H2** (TF-IDF is far cheaper) is the index-build and query time.

In [ ]:
from scipy.stats import wilcoxon
nd = {name: detail[name].set_index("reader")["NDCG@10"] for name in detail}

print("Paired Wilcoxon signed-rank on per-reader NDCG@10:")
for a,b in [("MiniLM","TF-IDF"), ("TF-IDF","Popularity"), ("MiniLM","Popularity")]:
    diff = nd[a] - nd[b]
    if (diff == 0).all():
        print(f"  {a:11} vs {b:11}: identical per-reader NDCG@10, test not applicable")
        continue
    st = wilcoxon(nd[a], nd[b])
    print(f"  {a:11} vs {b:11}: W={st.statistic:.0f}  p={st.pvalue:.2e}"
          f"  (median delta {np.median(diff):+.3f})")

print(f"\nH2 timing:")
print(f"  TF-IDF index build : {build_tfidf:6.1f}s   query total: {qt_tfidf:5.1f}s")
print(f"  MiniLM index build : {build_minilm:6.1f}s   query total: {qt_minilm:5.1f}s")
print(f"  build speedup (MiniLM / TF-IDF): {build_minilm/max(build_tfidf,1e-9):.0f}x")

In [ ]:
# Save everything so the numbers are reproducible and citable in the reports
summary.to_csv("data/results_summary.csv")
pd.concat(detail.values()).to_csv("data/results_per_reader.csv", index=False)
print("saved data/results_summary.csv and data/results_per_reader.csv")
summary

## What to put in the corrected reports
- **Report 2 (data):** the real corpus is the full arXiv snapshot (195,149 papers across q-bio.NC + cs.AI + cs.NE, of which the scoped neuro/cognitive library is 36,314). The earlier per-category table (cs.AI = 15,120) does not match the corpus (real cs.AI is ~173k) and is replaced by `corpus_meta.json`.
- **Report 6 (results):** replace the typed table with `results_summary.csv` produced above, and state that ground truth is category-based and independent of both models.
- **Report 7 (errors):** draw the failure cases from real disagreements between the two rankings on specific readers (inspect `results_per_reader.csv` for the lowest-NDCG readers).
- **Report 8 (conclusions):** report H1 with the Wilcoxon result, not just the mean gap, and H2 with the measured build/query times. Keep the honest limitation: category ground truth is a coarse proxy for real preference, and there are still no observed user interactions.

The result may now differ from the original (semantic did not necessarily win by 27 points once the key stopped being built the semantic way). Whatever the numbers say, they are now earned.